# Steam Analyzer

This project aims to analyze trends in Steams discount strategy and predict price on games in the future based on it.

[Dataset](https://www.kaggle.com/datasets/benjaminlundkvist/steam-sales-historical-dataset)


In [18]:
import pandas as pd
import asyncio
from playwright.async_api import async_playwright

async def get_app_id(game_name: str) -> str:
    url = f"https://steamdb.info/search/?a=app&q={game_name}"
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(url)
        # ждём пока таблица загрузится
        await page.wait_for_selector("table.table-products td:nth-child(2) a")
        link = await page.query_selector("table.table-products td:nth-child(2) a")
        href = await link.get_attribute("href")
        await browser.close()
        if href and "/app/" in href:
            return href.split("/")[2]
        return None

async def get_price_history(app_id: str, cc="us") -> pd.DataFrame:
    hist_url = f"https://steamdb.info/api/GetPriceHistory/?appid={app_id}&cc={cc}"
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(hist_url)
        content = await page.content()
        # json возвращается как текст
        text = await page.inner_text("pre")
        await browser.close()

    import json
    res = json.loads(text)
    data = []
    for entry in res.get("data", []):
        data.append((pd.to_datetime(entry["date"], unit="s"), entry["price"] / 100))
    df = pd.DataFrame(data, columns=["date", "price"])
    return df

async def get_game_price_history(game_name: str, start_date=None, end_date=None, cc="us"):
    app_id = await get_app_id(game_name)
    if not app_id:
        raise ValueError(f"Game '{game_name}' not found on SteamDB")
    df = await get_price_history(app_id, cc=cc)
    if start_date:
        df = df[df["date"] >= pd.to_datetime(start_date)]
    if end_date:
        df = df[df["date"] <= pd.to_datetime(end_date)]
    return df

# ====== Запуск ======
if __name__ == "__main__":
    game = "PEAK"
    df = asyncio.run(get_game_price_history(game, start_date="2025-06-01", end_date="2025-10-01", cc="us"))
    print(df.head())
    df.to_excel("price_history.xlsx", index=False)


RuntimeError: asyncio.run() cannot be called from a running event loop